# Top-k teacher logits on T4 with vLLM

Replaces the XLA/Torch TPU path, which failed six times on infrastructure
(`torch_xla`/`torch` ABI, no matching wheel, then no `libtpu` so PJRT reported
`CPU`).

Produces the same artefact: per **teacher** token, top-k token ids and their
log-probabilities, plus the token's byte offset so a 32k student can align by
byte span.

Two differences that matter on a T4:

- **Turing has no bf16.** The dtype must be `float16`.
- 8B in fp16 is 16 GB, which does not fit one T4 alongside KV cache, so the real
  run uses `tensor_parallel_size=2` over PCIe — measured working on Kaggle T4s.

Smoke first (4B, one GPU, 1M tokens, k=8). Stop if it is not well formed.


In [ ]:
import os, shutil, subprocess

smi = subprocess.run(['bash','-lc','nvidia-smi -L'], capture_output=True, text=True).stdout
print(smi)
assert 'T4' in smi, 'no T4 in this session; Settings -> Accelerator -> GPU T4 x2'

SMOKE = os.environ.get('SMOKE', '1') == '1'
TEACHER = 'Qwen/Qwen3-4B' if SMOKE else 'Qwen/Qwen3-8B'
K = 8 if SMOKE else 32
TP = 1 if SMOKE else 2
MAX_TOKENS = 1_000_000 if SMOKE else 22_700_000
WINDOW = 1024
print('teacher', TEACHER, 'k', K, 'tp', TP, 'budget', MAX_TOKENS)


In [ ]:
!pip install -q "vllm>=0.9" 2>&1 | tail -3
import vllm, torch
print('vllm', vllm.__version__, 'torch', torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'no CUDA device'


In [ ]:
import glob, struct
from pathlib import Path

def find(pattern):
    return sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))

TRAIN = find('train-*.bin') or find('*.bin')
assert TRAIN, 'attach sectionme/ferrisres-corpus-release-slice'
print('train shards:', len(TRAIN))

def read_u16(path):
    raw = Path(path).read_bytes()
    assert len(raw) % 2 == 0, f'{path}: odd byte count, not u16 ids'
    return list(struct.unpack('<%dH' % (len(raw) // 2), raw))

def byte_decode(ids):
    return bytes(i & 0xFF for i in ids).decode('utf-8', errors='strict')


In [ ]:
import json, time
import numpy as np
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

tok = AutoTokenizer.from_pretrained(TEACHER)
llm = LLM(model=TEACHER, dtype='float16', tensor_parallel_size=TP,
          max_model_len=WINDOW, gpu_memory_utilization=0.85, enforce_eager=True)
sp = SamplingParams(temperature=0.0, max_tokens=1, prompt_logprobs=K)

out_dir = Path('/kaggle/working/topk'); out_dir.mkdir(exist_ok=True)
for old in out_dir.glob('part-*.bin'):
    old.unlink()

total = 0; part = 0; parts = []; t0 = time.perf_counter()
for shard in TRAIN:
    if total >= MAX_TOKENS:
        break
    text = byte_decode(read_u16(shard))
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
    ids, offs = enc['input_ids'], enc['offset_mapping']
    print(shard, 'teacher tokens', len(ids))
    for i in range(0, len(ids), WINDOW):
        if total >= MAX_TOKENS:
            break
        chunk = ids[i:i + WINDOW]
        if len(chunk) < 2:
            break
        outs = llm.generate(prompt_token_ids=[chunk], sampling_params=sp)
        plp = outs[0].prompt_logprobs           # one entry per prompt position
        rows = []
        for pos in range(1, len(chunk)):
            entry = plp[pos]                     # dict {token_id: Logprob}
            if not entry:
                continue
            pairs = sorted(((t, lp.logprob) for t, lp in entry.items()),
                           key=lambda kv: kv[1], reverse=True)[:K]
            while len(pairs) < K:
                pairs.append((0, float('-inf')))
            rows.append((pairs, offs[i + pos][0]))
        if not rows:
            continue
        rec = np.zeros(len(rows), dtype=[('idx', '<i4', (K,)), ('val', '<f4', (K,)), ('off', '<i4')])
        rec['idx'] = np.array([[p[0] for p in r[0]] for r in rows], dtype='<i4')
        rec['val'] = np.array([[p[1] for p in r[0]] for r in rows], dtype='<f4')
        rec['off'] = np.array([r[1] for r in rows], dtype='<i4')
        assert (np.diff(rec['off']) >= 0).all(), 'offsets must be non-decreasing'
        name = f'part-{part:05d}.bin'
        rec.tofile(out_dir / name)
        parts.append({'part': name, 'tokens': int(len(rows)), 'shard': Path(shard).name})
        total += len(rows); part += 1
        if part % 50 == 0:
            print(total, 'tokens', round(time.perf_counter() - t0, 0), 's')

elapsed = time.perf_counter() - t0
manifest = {
    'teacher': TEACHER, 'k': K, 'dtype': 'float16', 'tp': TP, 'window': WINDOW,
    'record': 'idx:int32,val:float32,off:int32 (log-probabilities)',
    'tokens': total, 'parts': parts,
    'bytes': sum((out_dir / p['part']).stat().st_size for p in parts),
    'run_seconds': round(elapsed, 2),
    'tokens_per_s': round(total / elapsed, 1) if elapsed else 0.0,
    'smoke': SMOKE,
}
(out_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps({k: v for k, v in manifest.items() if k != 'parts'}, indent=2))
